# Aula 17 — Tool Use, Function Calling and Contracts

Nesta aula, o sistema deixa de apenas **responder** e passa a poder **solicitar uma ação externa estruturada**.

A ideia central é:

```text
responder
→ solicitar uma tool
→ validar argumentos
→ executar
→ observar resultado
→ produzir resposta final
```

O objetivo não é usar um agente pronto. É entender os componentes mínimos antes de adicionar mais autonomia.


## Objetivos

Ao final da aula, você deverá ser capaz de:

- distinguir resposta textual de tool call;
- explicar o que é um tool contract;
- definir e validar argumentos;
- separar tool request, execution, result e assistant response;
- localizar falhas por estágio;
- distinguir ferramentas read-only de state-changing;
- observar chamadas de ferramentas;
- explicar por que tools não são sinônimo de agentes.


## Glossário da aula

Conceitos centrais no **Glossário Vivo**:

**[Chamada de ferramenta](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#chamada-de-ferramenta) · [Chamada de função](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#chamada-de-função) · [Contrato de ferramenta](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#contrato-de-ferramenta) · [Schema de entrada](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#schema-de-entrada) · [Registro de ferramentas](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#registro-de-ferramentas) · [Efeito colateral](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#efeito-colateral) · [Gate de aprovação](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#gate-de-aprovação) · [Observabilidade de ferramentas](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#observabilidade-de-ferramentas)**

Versão em inglês: **[TIL Living Glossary — EN](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.en.md)**

> Use os links ao longo da aula quando quiser revisar a definição formal de um conceito sem interromper o fluxo do notebook.


## 1. Da resposta para a ação

Até agora, o sistema produzia essencialmente informação.

Exemplo:

```text
Pergunta:
Quanto é 12% de 850?

Resposta:
102
```

Agora queremos representar a ação explicitamente por meio de **[Tool Calling / Chamada de ferramenta](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#chamada-de-ferramenta)**:

```text
tool = calculate_percentage
arguments = {"value": 850, "percent": 12}
```

A decisão sobre **o que executar** e a execução real são coisas diferentes.


In [ ]:
from dataclasses import dataclass
from time import perf_counter
from typing import Any, Callable

import pandas as pd

print("Ambiente carregado.")


## 2. [Tool contract / Contrato de ferramenta](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#contrato-de-ferramenta)

Uma tool não deve ser apenas uma função escondida no código.

Vamos representar seu contrato com:

```text
name
description
input schema
side effect class
executor
```

O contrato torna a capacidade inspecionável antes da execução.

📚 Glossário: **[Contrato de ferramenta](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#contrato-de-ferramenta)** · **[Schema de entrada](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#schema-de-entrada)**


In [ ]:
@dataclass
class Tool:
    name: str
    description: str
    input_schema: dict
    side_effect_class: str
    executor: Callable[..., Any]


## 3. Três ferramentas locais

Usaremos ferramentas pequenas e determinísticas:

1. consultar um mini glossário;
2. calcular uma porcentagem;
3. consultar o status de uma aula.

Nenhuma delas depende de internet ou API externa.


In [ ]:
GLOSSARY = {
    "groundedness": "Grau em que uma resposta é sustentada pela evidência fornecida.",
    "factuality": "Grau em que uma afirmação corresponde aos fatos relevantes.",
    "tool calling": "Solicitação estruturada para executar uma ferramenta externa.",
}

LESSON_STATUS = {
    "14": "Available",
    "15": "Available",
    "16": "Available",
    "17": "Draft",
}

def lookup_glossary(term: str):
    key = term.strip().lower()
    return GLOSSARY.get(key, "Termo não encontrado.")

def calculate_percentage(value: float, percent: float):
    return value * percent / 100.0

def get_lesson_status(lesson_id: str):
    return LESSON_STATUS.get(str(lesson_id), "Unknown")


## 4. [Input schema / Schema de entrada](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#schema-de-entrada)

O schema descreve o que uma chamada aceita.

Aqui usaremos um schema didático simples com:

- `type`;
- `required`;
- `properties`.

A Aula 17 ensina o conceito sem depender de uma biblioteca específica de validação.

📚 Glossário: **[Schema de entrada](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#schema-de-entrada)**


In [ ]:
TOOLS = {
    "lookup_glossary": Tool(
        name="lookup_glossary",
        description="Consulta um termo no mini Glossário Vivo da aula.",
        input_schema={
            "type": "object",
            "required": ["term"],
            "properties": {"term": {"type": "string"}},
        },
        side_effect_class="read-only",
        executor=lookup_glossary,
    ),
    "calculate_percentage": Tool(
        name="calculate_percentage",
        description="Calcula uma porcentagem de um valor.",
        input_schema={
            "type": "object",
            "required": ["value", "percent"],
            "properties": {
                "value": {"type": "number"},
                "percent": {"type": "number"},
            },
        },
        side_effect_class="read-only",
        executor=calculate_percentage,
    ),
    "get_lesson_status": Tool(
        name="get_lesson_status",
        description="Consulta o status de uma aula do TIL.",
        input_schema={
            "type": "object",
            "required": ["lesson_id"],
            "properties": {"lesson_id": {"type": "string"}},
        },
        side_effect_class="read-only",
        executor=get_lesson_status,
    ),
}

registry_view = pd.DataFrame([
    {
        "tool": tool.name,
        "description": tool.description,
        "required": tool.input_schema["required"],
        "side_effect": tool.side_effect_class,
    }
    for tool in TOOLS.values()
])

display(registry_view)


## 5. Validation

Uma tool call não deve executar imediatamente.

Primeiro validamos:

```text
tool existe?
→ campos obrigatórios existem?
→ campos desconhecidos existem?
→ tipos estão corretos?
```

Isso separa **intenção proposta** de **ação permitida**.


In [ ]:
TYPE_MAP = {
    "string": str,
    "number": (int, float),
}

def validate_arguments(tool: Tool, arguments: dict):
    schema = tool.input_schema
    properties = schema["properties"]
    required = schema.get("required", [])

    missing = [name for name in required if name not in arguments]
    if missing:
        return False, f"missing required fields: {missing}"

    unknown = [name for name in arguments if name not in properties]
    if unknown:
        return False, f"unknown fields: {unknown}"

    for name, value in arguments.items():
        expected = properties[name]["type"]
        if not isinstance(value, TYPE_MAP[expected]):
            return False, f"{name} must be {expected}"

    return True, None


## 6. Uma chamada válida

Vamos observar o ciclo completo:

```text
request
→ validation
→ execution
→ result
```


In [ ]:
def run_tool(tool_name: str, arguments: dict):
    trace = {
        "tool_name": tool_name,
        "arguments": arguments,
        "selection_status": "pending",
        "validation_status": "not_run",
        "execution_status": "not_run",
        "result": None,
        "error_type": None,
        "latency_ms": None,
    }

    tool = TOOLS.get(tool_name)
    if tool is None:
        trace["selection_status"] = "failed"
        trace["error_type"] = "tool_selection_failure"
        return trace

    trace["selection_status"] = "ok"

    valid, error = validate_arguments(tool, arguments)
    if not valid:
        trace["validation_status"] = "failed"
        trace["error_type"] = "validation_failure"
        trace["result"] = error
        return trace

    trace["validation_status"] = "ok"

    start = perf_counter()
    try:
        result = tool.executor(**arguments)
        trace["execution_status"] = "ok"
        trace["result"] = result
    except Exception as exc:
        trace["execution_status"] = "failed"
        trace["error_type"] = "execution_failure"
        trace["result"] = repr(exc)
    finally:
        trace["latency_ms"] = (perf_counter() - start) * 1000

    return trace


In [ ]:
valid_trace = run_tool(
    "calculate_percentage",
    {"value": 850, "percent": 12},
)

pd.Series(valid_trace)


## 7. Tool result não é assistant response

A ferramenta retornou um valor.

```text
tool result = 102.0
```

Uma camada posterior poderia transformar isso em:

```text
"12% de 850 é 102."
```

Esses dois artefatos não devem ser confundidos.


In [ ]:
tool_result = valid_trace["result"]
assistant_response = f"12% de 850 é {tool_result:.0f}."

print("tool result:", tool_result)
print("assistant response:", assistant_response)


## 8. Failure taxonomy

Vamos separar falhas por estágio:

| Tipo | Significado |
|---|---|
| tool selection failure | ferramenta escolhida não existe ou é inadequada |
| argument generation failure | argumentos propostos representam incorretamente a intenção |
| validation failure | contrato rejeita a chamada |
| execution failure | chamada válida falha durante execução |
| result interpretation failure | resultado correto é interpretado incorretamente |

Assim como no RAG, dizer apenas **"tool use falhou"** é diagnóstico insuficiente.


## 9. Laboratório — localizando a falha

Observe as chamadas abaixo e identifique o primeiro estágio quebrado.


In [ ]:
failure_cases = [
    ("A", "unknown_tool", {"value": 100}),
    ("B", "calculate_percentage", {"value": 850}),
    ("C", "calculate_percentage", {"value": "850", "percent": 12}),
    ("D", "lookup_glossary", {"term": "groundedness"}),
]

failure_traces = []
for case, tool_name, arguments in failure_cases:
    trace = run_tool(tool_name, arguments)
    trace["case"] = case
    failure_traces.append(trace)

display(pd.DataFrame(failure_traces)[
    ["case", "tool_name", "selection_status", "validation_status",
     "execution_status", "error_type", "result"]
])


### 9.1 Argument generation failure é diferente de validation failure

Considere:

```text
Usuário:
Calcule 12% de 850.

Chamada proposta:
calculate_percentage(value=12, percent=850)
```

Os tipos são válidos.

O schema pode aceitar a chamada.

Mas os argumentos **não representam corretamente a intenção**.

Isso é um **argument generation failure** que a validação estrutural simples não detecta.


In [ ]:
semantically_wrong = run_tool(
    "calculate_percentage",
    {"value": 12, "percent": 850},
)

pd.Series(semantically_wrong)


## 10. [Side effects / Efeitos colaterais](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#efeito-colateral) e safety

Nem toda ferramenta tem o mesmo risco.

Uma classificação útil:

```text
read-only
→ consulta sem alterar estado

state-changing
→ modifica estado externo

high-impact
→ pode produzir consequência relevante ou difícil de reverter
```

Quanto maior o impacto, maior a necessidade de validação, autorização e supervisão.

📚 Glossário: **[Efeito colateral](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#efeito-colateral)** · **[Gate de aprovação](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#gate-de-aprovação)**


In [ ]:
side_effect_examples = pd.DataFrame([
    {"tool": "lookup_glossary", "class": "read-only", "approval": "not usually required"},
    {"tool": "update_profile", "class": "state-changing", "approval": "context dependent"},
    {"tool": "send_message", "class": "state-changing", "approval": "recommended"},
    {"tool": "execute_payment", "class": "high-impact", "approval": "required"},
])

display(side_effect_examples)


## 11. [Approval gate / Gate de aprovação](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#gate-de-aprovação)

Um approval gate separa:

```text
ação proposta
→ autorização
→ execução
```

Nesta aula não executaremos ações irreversíveis.

O conceito será importante quando passarmos de ferramentas locais para workflows e sistemas agentes.

📚 Glossário: **[Gate de aprovação](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#gate-de-aprovação)**


> **Governar antes da execução** significa submeter a ação proposta a regras de validação, autorização e política antes de permitir que ela produza efeitos reais.


## 12. [Observabilidade de ferramentas](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#observabilidade-de-ferramentas)

Um log útil deveria permitir reconstruir o que aconteceu.

Exemplo:

```text
tool_name
arguments
selection_status
validation_status
execution_status
latency_ms
result_summary
error_type
side_effect_class
```

Não queremos apenas saber se "deu certo". Queremos saber **onde e por quê**.

📚 Glossário: **[Observabilidade de ferramentas](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#observabilidade-de-ferramentas)**


In [ ]:
observability_log = []

for tool_name, arguments in [
    ("calculate_percentage", {"value": 500, "percent": 10}),
    ("lookup_glossary", {"term": "factuality"}),
    ("get_lesson_status", {"lesson_id": "16"}),
]:
    trace = run_tool(tool_name, arguments)
    trace["side_effect_class"] = TOOLS[tool_name].side_effect_class
    observability_log.append(trace)

display(pd.DataFrame(observability_log))


## 13. Função direta vs tool estruturada

Uma chamada direta:

```python
calculate_percentage(850, 12)
```

é mais simples.

O **[Tool Registry / Registro de ferramentas](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#registro-de-ferramentas)** acrescenta:

- discovery;
- contrato;
- validação;
- logs;
- política de execução;
- base para workflows.

A pergunta do TIL permanece:

> **A complexidade adicional trouxe capacidade necessária?**


In [ ]:
direct_result = calculate_percentage(850, 12)
registry_result = run_tool("calculate_percentage", {"value": 850, "percent": 12})["result"]

comparison = pd.DataFrame([
    {"approach": "direct function", "result": direct_result, "contract": False, "validation": False, "trace": False},
    {"approach": "tool registry", "result": registry_result, "contract": True, "validation": True, "trace": True},
])

display(comparison)


## 14. Exercício 1 — Desenhe um tool contract

Projete um contrato para:

```text
convert_temperature(value, from_unit, to_unit)
```

Defina campos obrigatórios e tipos.


In [ ]:
# Sua resposta aqui
temperature_schema = {
    # ...
}


### Dica

Use a mesma estrutura dos schemas anteriores:

```text
type
required
properties
```


In [ ]:
temperature_schema_solution = {
    "type": "object",
    "required": ["value", "from_unit", "to_unit"],
    "properties": {
        "value": {"type": "number"},
        "from_unit": {"type": "string"},
        "to_unit": {"type": "string"},
    },
}

temperature_schema_solution


## 15. Exercício 2 — Quebre o schema

Crie uma chamada inválida para `calculate_percentage` e explique qual validação deve falhar.


In [ ]:
# Sua resposta aqui
invalid_call = {
    # ...
}


### Dica

Experimente remover um campo obrigatório ou passar uma string onde o schema espera número.


In [ ]:
invalid_call_solution = {"value": "850", "percent": 12}
invalid_trace_solution = run_tool("calculate_percentage", invalid_call_solution)

pd.Series(invalid_trace_solution)


## 16. Exercício 3 — Localize a falha

Considere:

```text
Usuário: "Calcule 12% de 850"
Tool: calculate_percentage
Arguments: {"value": 12, "percent": 850}
Validation: OK
Execution: OK
Result: 102
```

Qual categoria descreve melhor o problema?


### Dica

O schema consegue verificar estrutura e tipos, mas não necessariamente se os argumentos representam corretamente a intenção do usuário.


In [ ]:
exercise_3_solution = "argument generation failure"
print(exercise_3_solution)


## 17. Exercício 4 — Read-only ou state-changing?

Classifique:

- consultar glossário;
- enviar e-mail;
- atualizar perfil;
- consultar status de aula;
- executar pagamento.


In [ ]:
exercise_4_solution = pd.DataFrame([
    {"action": "consultar glossário", "class": "read-only"},
    {"action": "enviar e-mail", "class": "state-changing"},
    {"action": "atualizar perfil", "class": "state-changing"},
    {"action": "consultar status de aula", "class": "read-only"},
    {"action": "executar pagamento", "class": "high-impact"},
])

display(exercise_4_solution)


## 18. Exercício 5 — Uma tool é necessária?

Nem toda capacidade precisa virar uma tool.

Neste exercício, sua tarefa é escolher a **arquitetura mínima suficiente** para cada situação:

```text
direct_answer
direct_function
structured_tool
```

Quando necessário, revise **[Registro de ferramentas](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#registro-de-ferramentas)**, **[Efeito colateral](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#efeito-colateral)** e **[Gate de aprovação](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#gate-de-aprovação)** no Glossário Vivo:

Use estes critérios:

| Critério | Pergunta |
|---|---|
| **Complexidade** | Preciso realmente adicionar uma camada de execução? |
| **Determinismo** | Uma função local simples resolve? |
| **Observabilidade** | Preciso registrar chamada, argumentos, resultado e erro? |
| **Reutilização** | Outros componentes precisarão descobrir e usar essa capacidade? |
| **Risco** | A ação altera estado ou produz efeito externo? |
| **Governança** | Preciso de validação, autorização ou approval gate? |

A regra do TIL permanece:

> **Escolha a solução mais simples que entregue a capacidade necessária.**


In [ ]:
architecture_cases = pd.DataFrame([
    {
        "case": "A",
        "situation": "Explique a diferença entre precision e recall.",
        "risk": "low",
        "state_change": False,
        "needs_external_capability": False,
    },
    {
        "case": "B",
        "situation": "Calcule 12% de 850.",
        "risk": "low",
        "state_change": False,
        "needs_external_capability": False,
    },
    {
        "case": "C",
        "situation": "Consulte o status da Aula 16 no catálogo do TIL.",
        "risk": "low",
        "state_change": False,
        "needs_external_capability": True,
    },
    {
        "case": "D",
        "situation": "Consulte o termo 'groundedness' no Glossário Vivo.",
        "risk": "low",
        "state_change": False,
        "needs_external_capability": True,
    },
    {
        "case": "E",
        "situation": "Converta este texto para letras maiúsculas.",
        "risk": "low",
        "state_change": False,
        "needs_external_capability": False,
    },
    {
        "case": "F",
        "situation": "Envie esta mensagem para um canal externo.",
        "risk": "medium",
        "state_change": True,
        "needs_external_capability": True,
    },
    {
        "case": "G",
        "situation": "Apague um arquivo do sistema.",
        "risk": "high",
        "state_change": True,
        "needs_external_capability": True,
    },
    {
        "case": "H",
        "situation": "Qual é a capital do Brasil?",
        "risk": "low",
        "state_change": False,
        "needs_external_capability": False,
    },
])

display(architecture_cases)


### Sua tarefa

Para cada caso A–H, escolha uma das opções:

- `direct_answer`
- `direct_function`
- `structured_tool`

Depois escreva uma justificativa curta.

Não existe a obrigação de usar tool só porque uma função poderia ser exposta como tool. O objetivo é identificar **quando o contrato, a observabilidade e a governança justificam a complexidade adicional**.


In [ ]:
# Sua resposta aqui
#
# Exemplo:
# architecture_choice = {
#     "A": "direct_answer",
#     "B": "...",
# }
#
# justification = {
#     "A": "Não há necessidade de executar uma capacidade externa.",
# }


### Dica

Pense na seguinte sequência:

```text
Só precisa responder?
→ direct_answer

Precisa executar uma transformação local, simples e determinística?
→ direct_function

Precisa expor uma capacidade reutilizável, observável, validável
ou com efeito externo?
→ structured_tool
```

Para ações com side effect, considere também approval gates.


In [ ]:
architecture_choice = {
    "A": "direct_answer",
    "B": "direct_function",
    "C": "structured_tool",
    "D": "structured_tool",
    "E": "direct_function",
    "F": "structured_tool",
    "G": "structured_tool",
    "H": "direct_answer",
}

justification = {
    "A": "É uma explicação conceitual; nenhuma execução externa é necessária.",
    "B": "É uma operação determinística simples que uma função local resolve.",
    "C": "Consulta uma capacidade externa ao modelo e se beneficia de contrato e observabilidade.",
    "D": "Consulta uma fonte estruturada; a tool permite reutilização e rastreabilidade.",
    "E": "Transformação local simples, sem necessidade de registry ou governança adicional.",
    "F": "Há efeito externo; contrato, validação, logging e controle de execução são importantes.",
    "G": "Há efeito destrutivo; precisa de tool estruturada e approval gate antes da execução.",
    "H": "Conhecimento factual simples; não há necessidade de ação externa neste exercício.",
}

solution_table = architecture_cases[["case", "situation"]].copy()
solution_table["choice"] = solution_table["case"].map(architecture_choice)
solution_table["why"] = solution_table["case"].map(justification)

display(solution_table)


### Interpretação

O exercício não procura uma regra universal de que determinada tarefa **sempre** deve usar uma arquitetura específica.

O contexto importa.

Por exemplo, calcular uma porcentagem pode ser apenas uma função local em um notebook. Mas, se a mesma capacidade precisar ser descoberta dinamicamente por vários modelos, validada, auditada e reutilizada por workflows, ela pode ser exposta como tool.

O ponto central é:

```text
tool
≠
arquitetura automaticamente melhor
```

Uma tool acrescenta:

```text
contrato
+ validação
+ discovery
+ observabilidade
+ reutilização
+ governança
```

mas também acrescenta:

```text
código
+ estados
+ latência
+ failure modes
+ custo de governança
```

Portanto:

> **A complexidade deve ser proporcional à capacidade, ao risco e à necessidade de controle.**


## 19. Ponte para MCP

Nesta aula construímos:

```text
função Python
→ tool contract
→ Tool Registry
```

Ainda não usamos MCP.

A progressão planejada é:

```text
Tool Registry
→ deterministic workflow
→ MCP server
→ agentic system
```

MCP não será apresentado como sinônimo de agente. Ele será estudado como uma camada padronizada para expor e descobrir capacidades.

Antes de seguir, revise no Glossário Vivo os conceitos que serão reutilizados nessa transição:

**[Registro de ferramentas](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#registro-de-ferramentas) · [Contrato de ferramenta](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#contrato-de-ferramenta) · [Observabilidade de ferramentas](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#observabilidade-de-ferramentas)**


## 20. Reprodutibilidade

Esta aula foi desenhada para:

```text
Internet OFF
GPU OFF
sem API externa
sem side effects reais
```

O laboratório usa apenas funções locais e dados didáticos.


## 21. Síntese

A principal mudança desta aula é arquitetural:

```text
texto gerado
→ solicitação estruturada
→ validação
→ execução
→ resultado observável
```

Uma tool não é apenas uma função que um modelo "chama".

É uma capacidade externa exposta por um **contrato**, governada antes da execução — quer dizer: submetida a regras de validação, autorização e política antes de executar — e observada durante seu uso.

Na próxima aula, vamos combinar tools em **Deterministic Workflows**.



---

## Continue no TIL

← **[Anterior: Aula 16 — Retrieval-Augmented Generation (RAG)](https://www.kaggle.com/code/pedrogentil/til-16-retrieval-augmented-generation)** &nbsp;&nbsp;|&nbsp;&nbsp; 🏠 **[Apresentação do curso](https://www.kaggle.com/code/pedrogentil/text-intelligence-lab-course)** &nbsp;&nbsp;|&nbsp;&nbsp; **[Próxima: Aula 18 — Deterministic Workflows — em preparação](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/ROADMAP.md)** →
